In [1]:
import sys, subprocess, os, ctypes, glob, importlib, site

print("💉 Injecting NVIDIA JIT Linker...")
lib_path = "/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib/libnvJitLink.so*"
found_libs = glob.glob(lib_path)
if found_libs:
    ctypes.cdll.LoadLibrary(found_libs[0])
    os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

print("📦 Installing Offline AI Tools...")
wheel_files = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if (f.endswith(".whl") or f.endswith(".tar.gz")) and not any(x in f.lower() for x in ["numpy", "pillow", "scipy"]):
            wheel_files.append(os.path.join(root, f))

if wheel_files:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheel_files)
    importlib.reload(site)
    print("✅ Environment is bulletproof.")

💉 Injecting NVIDIA JIT Linker...
📦 Installing Offline AI Tools...
✅ Environment is bulletproof.


In [2]:
from unsloth import FastLanguageModel 
import torch

MODEL_ID = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

print("⏳ Loading the 30-Billion parameter brain into RTX Pro 6000...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    dtype=torch.bfloat16, 
    load_in_4bit=True,    
    trust_remote_code=True
)
print("✅ Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[bitsandbytes.cextension|ERROR]bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Loading the 30-Billion parameter brain into RTX Pro 6000...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.10: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Nemotron_H does not support SDPA - switching to fast eager.


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

NemotronHForCausalLM LOAD REPORT from: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Key                                                               | Status     |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

RuntimeError: We encountered some issues during automatic conversion of the weights. For details look at the `CONVERSION` entries of the above report!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

print("🔗 Attaching LoRA Adapters...")
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

print("🛡️ Deploying Precision Shields...")
def shield_hook(m, args, output):
    def cast(x): return x.to(torch.bfloat16) if isinstance(x, torch.Tensor) and x.is_floating_point() else x
    return cast(output) if isinstance(output, torch.Tensor) else tuple(cast(o) for o in output)

def input_bouncer(m, args):
    if isinstance(args[0], torch.Tensor) and args[0].dtype == torch.uint8:
        return (args[0].to(torch.bfloat16),) + args[1:]
    return args

for name, module in model.named_modules():
    if any(k in name.lower() for k in ["expert", "gate", "proj"]):
        module.register_forward_pre_hook(input_bouncer)
        module.register_forward_hook(shield_hook)

print("🚀 Starting 60-step training loop...")
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_dataset,
    dataset_text_field="text", max_seq_length=1024,
    args=TrainingArguments(
        output_dir="./nemotron_adapter", per_device_train_batch_size=2,
        gradient_accumulation_steps=4, optim="adamw_8bit",
        learning_rate=2e-4, bf16=True, max_steps=60, report_to="none"
    ),
)
trainer.train()
print("✅ Training complete!")

In [ ]:
print("🧪 Switching to Inference Mode...")
FastLanguageModel.for_inference(model)

predictions = []
print(f"🚀 Generating answers for {len(test_df)} questions...")

for _, row in test_df.iterrows():
    prompt = f"User: Solve this reasoning task.\nTask: {row['question']}\n\nAssistant: "
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    decoded = tokenizer.batch_decode(outputs)[0]
    
    # Extract only the assistant's final boxed answer
    final_answer = decoded.split("Assistant: ")[-1].strip()
    predictions.append(final_answer)

# Create the final leaderboard file
submission_csv = pd.DataFrame({"id": test_df["id"], "prediction": predictions})
submission_csv.to_csv("submission.csv", index=False)
print("🏁 submission.csv created!")